# 02 - Embeddings Profundos com CelebA Align

Neste notebook vamos gerar embeddings com `ResNet50`, visualizar a distribuicao em 2D e aplicar `K-Means` como metodo final classico.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from PIL import Image
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet50_Weights, resnet50

try:
    import umap
except ImportError:
    umap = None

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent
DATASET_ROOT = Path('/home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/data/raw/img_align_celeba')
OUTPUT_PATH = PROJECT_ROOT / 'artifacts' / 'embeddings' / 'celeba_align_deep_resnet50.parquet'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
MAX_SAMPLES = 500
N_CLUSTERS = 10

for folder in [OUTPUT_PATH.parent, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DATASET_ROOT

PosixPath('/home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/data/raw/img_align_celeba')

In [3]:
def build_align_manifest(dataset_root, max_samples=None):
    dataset_root = Path(dataset_root)
    images = sorted(dataset_root.glob('*.jpg'))
    if not images:
        raise FileNotFoundError('Nenhuma imagem .jpg foi encontrada em img_align_celeba.')
    if max_samples:
        images = images[:max_samples]
    return pd.DataFrame({
        'image_name': [path.name for path in images],
        'image_path': [str(path) for path in images],
    })

class AlignCelebADataset(Dataset):
    def __init__(self, dataset_root, image_size=224, max_samples=None):
        self.samples = build_align_manifest(dataset_root, max_samples=max_samples)
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        row = self.samples.iloc[index]
        image = Image.open(row['image_path']).convert('RGB')
        return {
            'image': self.transform(image),
            'image_name': row['image_name'],
            'image_path': row['image_path'],
        }

def build_resnet50_embedding_model():
    model = resnet50(weights=ResNet50_Weights.DEFAULT)
    model.fc = nn.Identity()
    model.eval()
    return model

def extract_deep_embeddings(dataset_root, output_path, batch_size=32, num_workers=0, max_samples=None, device=None):
    dataset = AlignCelebADataset(dataset_root=dataset_root, max_samples=max_samples)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    resolved_device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    model = build_resnet50_embedding_model().to(resolved_device)

    rows = []
    with torch.no_grad():
        for batch in dataloader:
            images = batch['image'].to(resolved_device)
            embeddings = model(images).cpu().numpy()
            image_names = batch['image_name']
            image_paths = batch['image_path']
            for embedding, image_name, image_path in zip(embeddings, image_names, image_paths):
                row = {'image_name': image_name, 'image_path': image_path}
                for index, value in enumerate(embedding):
                    row[f'f_{index:04d}'] = float(value)
                rows.append(row)

    frame = pd.DataFrame(rows)
    frame.to_parquet(output_path, index=False)
    return frame

def feature_matrix(frame):
    return frame.drop(columns=['image_name', 'image_path'])

def reduce_embeddings(frame, method='pca', random_state=42):
    features = feature_matrix(frame)
    if method == 'pca':
        reducer = PCA(n_components=2, random_state=random_state)
    elif method == 'tsne':
        reducer = TSNE(n_components=2, random_state=random_state, init='pca')
    elif method == 'umap':
        if umap is None:
            raise ImportError("Instale 'umap-learn' para usar UMAP.")
        reducer = umap.UMAP(n_components=2, random_state=random_state)
    else:
        raise ValueError(f'Metodo desconhecido: {method}')

    reduced = reducer.fit_transform(features)
    result = frame[['image_name', 'image_path']].copy()
    result['x'] = reduced[:, 0]
    result['y'] = reduced[:, 1]
    result['method'] = method
    return result

def cluster_embeddings(frame, n_clusters=10, random_state=42):
    features = feature_matrix(frame)
    model = KMeans(n_clusters=n_clusters, random_state=random_state, n_init='auto')
    clusters = model.fit_predict(features)
    score = silhouette_score(features, clusters)
    clustered = frame[['image_name', 'image_path']].copy()
    clustered['cluster'] = clusters
    return clustered, score

def save_scatter_plot(frame_2d, output_path, title, hue_column='cluster'):
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=frame_2d, x='x', y='y', hue=hue_column, palette='tab10', s=30, alpha=0.75)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(output_path, dpi=180)
    plt.close()
    return output_path

## Extracao dos embeddings profundos

In [4]:
deep_df = extract_deep_embeddings(
    dataset_root=DATASET_ROOT,
    output_path=OUTPUT_PATH,
    max_samples=MAX_SAMPLES,
)
deep_df.head()

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /home/arthur/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:08<00:00, 12.6MB/s]


,image_name,image_path,f_0000,f_0001,f_0002,f_0003,f_0004,f_0005,f_0006,f_0007,...,f_2038,f_2039,f_2040,f_2041,f_2042,f_2043,f_2044,f_2045,f_2046,f_2047
0,000001.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,0.115439,0.000000,1.770863,0.000000,0.003611,0.00000,0.000000,1.845979,...,0.027631,0.000000,0.134002,0.024868,0.000000,0.000000,0.002578,0.017375,0.000000,0.045620
1,000002.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,0.000000,0.000000,0.003530,0.080691,0.000000,0.18496,0.137095,1.308235,...,0.000000,0.006563,0.043305,0.097198,0.015327,0.000000,0.000000,0.000000,0.071328,0.000000
2,000003.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,0.000000,0.000000,0.041407,0.000000,0.003639,0.00000,0.012828,0.905839,...,0.000000,0.000000,0.367652,0.417643,0.114686,0.000000,0.007574,0.034192,0.000000,0.008860
3,000004.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,0.000000,0.326497,0.133340,0.000000,0.000000,0.00000,0.012493,2.536354,...,0.000000,0.000000,0.011816,0.053964,0.000000,0.000000,0.143138,0.010553,0.000000,0.020305
4,000005.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,0.000056,0.068330,0.672592,0.252419,0.399458,0.00000,0.000000,0.459869,...,0.023714,0.046736,0.044074,0.203501,0.000000,0.008572,0.000000,0.000000,0.532222,0.093290


## Reducao para 2D e clustering

In [5]:
clustered_deep_df, deep_silhouette = cluster_embeddings(deep_df, n_clusters=N_CLUSTERS)
deep_2d = reduce_embeddings(deep_df, method='pca').merge(clustered_deep_df, on=['image_name', 'image_path'])
deep_2d.head()

,image_name,image_path,x,y,method,cluster
0,000001.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,7.105931,-1.979785,pca,1
1,000002.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,-1.396491,-2.301612,pca,8
2,000003.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,-0.244475,-5.061518,pca,8
3,000004.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,2.483761,3.989598,pca,6
4,000005.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,1.971561,-4.138616,pca,8


In [6]:
figure_path = FIGURES_DIR / 'celeba_align_deep_pca_clusters.png'
save_scatter_plot(deep_2d, figure_path, 'CelebA Align - Deep Embeddings + K-Means')
figure_path

PosixPath('/home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/reports/figures/celeba_align_deep_pca_clusters.png')

In [7]:
print('Silhouette score (deep):', round(deep_silhouette, 4))

Silhouette score (deep): 0.0481
